# Phase 6 — BRID controlled synthetic case

`SYNTHETIC_TEST_ONLY`

Build a source-grounded, reviewable matching contract without activating filters or matching.

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Local validation mode')


In [ ]:
NOTEBOOK_NAME = '11_PHASE_6_BRID_CONTROLLED_CASE.ipynb'
DEVOTEAM_PARENT = Path('/content/drive/MyDrive/Devoteam internship')
REQUIRED_FILES = (
    Path('README_START_HERE.md'),
    Path('scripts/run_phase6_brid_case.py'),
    Path('requirements/phase6_brid.txt'),
    Path('tests/test_phase6_brid_contract.py'),
    Path('human_inputs/phase6/OPPORTUNITY_INPUT_BRID_SYNTHETIC.pdf'),
)

def valid_project_root(path):
    return path.is_dir() and all((path / item).is_file() for item in REQUIRED_FILES)

if IN_COLAB:
    if not DEVOTEAM_PARENT.is_dir():
        raise FileNotFoundError(
            'Expected Drive folder is not mounted: '
            f'{DEVOTEAM_PARENT}. Confirm that MyDrive/Devoteam internship exists.'
        )

    # The verified project is one level below "Devoteam internship".
    # This is deliberately shallow: never scan the complete mounted Drive.
    candidates = [DEVOTEAM_PARENT]
    candidates.extend(
        path for path in DEVOTEAM_PARENT.iterdir()
        if path.is_dir() and not path.name.startswith('.')
    )
    valid_candidates = sorted(
        {path.resolve() for path in candidates if valid_project_root(path)},
        key=str,
    )
    notebook_candidates = [
        path for path in valid_candidates if (path / NOTEBOOK_NAME).is_file()
    ]

    if len(notebook_candidates) == 1:
        PROJECT_ROOT = notebook_candidates[0]
    elif len(valid_candidates) == 1:
        PROJECT_ROOT = valid_candidates[0]
    else:
        inspected = [path.name for path in candidates]
        raise RuntimeError(
            'Could not select one Phase 6 project folder. '
            f'Valid candidates: {[str(p) for p in valid_candidates]}. '
            f'Inspected direct folders: {inspected}'
        )
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if not valid_project_root(PROJECT_ROOT):
        raise FileNotFoundError(
            f'Local validation must start in the project root; got {PROJECT_ROOT}'
        )

missing = [
    str(item) for item in REQUIRED_FILES
    if not (PROJECT_ROOT / item).is_file()
]
assert not missing, f'Phase 6 project contract is incomplete: {missing}'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('Phase 6 input contract: PASS')


In [ ]:
import subprocess, sys
if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements/phase6_brid.txt')],
        check=True
    )
else:
    print('Dependencies supplied by the validated local environment')


In [ ]:
import json, subprocess, sys
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'run_phase6_brid_case.py')],
    cwd=PROJECT_ROOT, text=True, capture_output=True
)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError('Phase 6 BRID execution failed')


In [ ]:
test = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', 'tests/test_phase6_brid_contract.py'],
    cwd=PROJECT_ROOT, text=True, capture_output=True
)
print(test.stdout)
if test.returncode:
    print(test.stderr)
    raise RuntimeError('Phase 6 BRID tests failed')


In [ ]:
manifests = list((PROJECT_ROOT / 'data' / 'opportunities').glob('*/phase6_brid_controlled_case_v2/PHASE_6_BRID_MANIFEST.json'))
if len(manifests) != 1:
    raise AssertionError(f'Expected one BRID manifest, found {len(manifests)}')
manifest = json.loads(manifests[0].read_text(encoding='utf-8'))
display(manifest)
assert manifest['status'] == 'TECHNICAL_PASS_READY_FOR_HUMAN_REVIEW'
print('\nTECHNICAL_PASS_READY_FOR_HUMAN_REVIEW')
print('Next: review BRID_PHASE_6_REVIEW.xlsx. Matching remains blocked.')
